# 04 · Prompt und Permutationen


In [1]:
# --- Bootstrap: Projekt-Wurzel finden und importierbar machen ---
import sys, pathlib

PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])


PROJECT_ROOT: <Projektordner>/analysis
Python: 3.12.13


In [2]:
from src.data.registry import load_samples
from src.utils.config import named_config

samples, _ = load_samples(named_config("datasets", "medqa"), "test", limit=3)
sample = samples[0]
print(sample.sample_id, "| gold_index =", sample.gold_index, "(", sample.gold_label, ")")


test-00000 | gold_index = 1 ( B )


## Prompt

Die eingefrorene Zero-Shot-Vorlage `thesis-mcq-v2` steht in `configs/experiments/thesis.yaml`.


In [3]:
from src.prompts.mcq import MCQPrompt, PromptConfig
from src.utils.config import load_yaml

experiment = load_yaml("configs/experiments/thesis.yaml")
prompt = MCQPrompt(PromptConfig.from_mapping(experiment["prompt"]))

text = prompt.user_content(sample.question, sample.options)
print(text)


Select the single best answer. Respond with exactly one option ID: A, B, C, or D.

Question:
A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Answer:


## Zyklische Permutationen

Vier Verschiebungen ändern die Positionen A–D. `display_to_content` bildet Anzeige auf Inhalt ab, `content_to_display` umgekehrt. `gold_display_index` folgt der richtigen Antwort.


In [4]:
from src.permutations.cyclic import cyclic_permutations

perms = cyclic_permutations(sample)
print(f"Anzahl Permutationen: {len(perms)}\n")
for p in perms:
    print(f"perm {p.permutation_id}: "
          f"display->content={p.display_to_content} | "
          f"gold erscheint als '{p.gold_display_label}' (Anzeige-Index {p.gold_display_index})")


Anzahl Permutationen: 4

perm 0: display->content=(0, 1, 2, 3) | gold erscheint als 'B' (Anzeige-Index 1)
perm 1: display->content=(1, 2, 3, 0) | gold erscheint als 'A' (Anzeige-Index 0)
perm 2: display->content=(2, 3, 0, 1) | gold erscheint als 'D' (Anzeige-Index 3)
perm 3: display->content=(3, 0, 1, 2) | gold erscheint als 'C' (Anzeige-Index 2)


In [5]:
# Konkret: wie sieht Permutation 1 fuer das Modell aus?
p = perms[1]
print("Angezeigte Reihenfolge (perm 1):")
for d, opt in enumerate(p.displayed_options):
    is_gold = (d == p.gold_display_index)
    print(f"  {'ABCD'[d]}) {opt[:70]}{'   <-- richtig' if is_gold else ''}")


Angezeigte Reihenfolge (perm 1):
  A) Tell the attending that he cannot fail to disclose this mistake   <-- richtig
  B) Report the physician to the ethics committee
  C) Refuse to dictate the operative report
  D) Disclose the error to the patient and put it in the operative report


## Mapping prüfen

Inhaltsbasierte Antworten sollten über alle vier Varianten konsistent bleiben. Die Permutationen dienen der Bias-Messung und der PriDe-Priorschätzung.


In [6]:
# Sanity-Check: content_to_display ist die Umkehrung von display_to_content
for p in perms:
    for c in range(4):
        d = p.content_to_display[c]
        assert p.display_to_content[d] == c
print("OK: die beiden Abbildungen sind konsistente Umkehrungen.")


OK: die beiden Abbildungen sind konsistente Umkehrungen.
